# Inference on New Vestibular Schwannoma Cases

Segments new VS MRI scans with the models trained in `01_five_fold_cross_validation.ipynb` and saves the predicted tumor masks as NIfTI. It runs a **5-fold soft-voting ensemble**: all five fold models for the chosen architecture (UNet, DynUNet, or SegMamba) are run on the scan, their softmax probabilities averaged, and the average decoded into a mask (argmax + keep-largest-component). This usually beats any single fold.

## Preprocessing parity

A model gives correct output only when inference preprocesses the input exactly as training did. Three settings must match: `apply_reorder` (RAS+ reorientation), `target_spacing` (resample spacing), and `normalization` (foreground Z-normalization). A mismatch does not raise an error; it silently degrades the prediction. We get parity for free by loading the `PatchConfig` that notebook 01 saved at training time.

## 1. Environment setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import torchio as tio

from fastMONAI.vision_all import *
from fastai.learner import load_learner

# Resolve paths against the repo root so they work from any launch dir.
import fastMONAI
REPO_ROOT = Path(fastMONAI.__file__).resolve().parent.parent
os.chdir(REPO_ROOT / "research" / "vestibular_schwannoma")

## 2. Configuration

In [ ]:
MODEL_KEY = "unet"          # "unet" | "dynunet" | "segmamba"

# leave empty to auto-discover from MLflow.
FOLD_LEARNER_PATHS = {}

NEW_CASES = [
    "../nii_data/CASE_ID", 
]
OUTPUT_DIR = "inference_predictions"

USE_TTA = True     # 8-flip test-time augmentation (more robust, ~8x slower)
USE_AMP = True     # mixed precision on CUDA (ignored on CPU)

# Patch config: pulled from the model's MLflow run by default; set CONFIG_JSON to a local inference_patch_config.json to override.
CONFIG_JSON = None

print(f"Model: {MODEL_KEY} | mode: 5-fold soft-voting ensemble")
print(f"Cases to segment: {len(NEW_CASES)}")

### Locate the fold checkpoints

In [ ]:
# find_fold_learners (fastMONAI.utils) discovers one best_learner.pkl per fold from MLflow.
if not FOLD_LEARNER_PATHS:
    FOLD_LEARNER_PATHS = find_fold_learners(f"vs5f_{MODEL_KEY}")

print("Fold checkpoints:", {k: str(v) for k, v in sorted(FOLD_LEARNER_PATHS.items())} or "(none)")

## 3. Load the inference configuration

`PatchConfig` describes how patches are sampled and aggregated and how the raw input is preprocessed. We load the exact config notebook 01 saved at training time (from the model's MLflow run by default, or a local `inference_patch_config.json`) and rebuild `PatchConfig` from it.

In [ ]:
# Load the patch config notebook 01 saved: the source of truth for preprocessing and patch sampling.
if CONFIG_JSON and Path(CONFIG_JSON).exists():
    config_path = CONFIG_JSON
    print(f"Patch config: local file {config_path}")
else:
    _cfg = find_fold_learners(f"vs5f_{MODEL_KEY}",
                              artifact_path="config/inference_patch_config.json")
    if _cfg:
        config_path = next(iter(_cfg.values()))   # identical across folds; take any
        print(f"Patch config: MLflow run for 'vs5f_{MODEL_KEY}'")
    elif Path("inference_patch_config.json").exists():
        config_path = "inference_patch_config.json"
        print("Patch config: local inference_patch_config.json (no MLflow artifact found)")
    else:
        raise FileNotFoundError(
            "No patch config found. Run notebook 01 (it logs the config to MLflow and writes a "
            "local copy), or set CONFIG_JSON to a local inference_patch_config.json.")

# normalization travels inside the config; patch_inference applies the same ZNormalization.
patch_config = PatchConfig(**load_patch_variables(config_path))
print(patch_config)

## 4. Load the models

We load the five fold learners for `MODEL_KEY` into a list; each carries its own weights and preprocessing. `load_learner` uses Python `pickle`, which can execute arbitrary code, so only load checkpoints you trust.

In [ ]:
def _load_learner(pkl_path):
    """Load an exported fastai learner onto GPU if available, else CPU; set eval mode."""
    learn = load_learner(pkl_path, cpu=not torch.cuda.is_available())
    learn.model.eval()
    return learn


# Load the fold learners; load_learner uses pickle, so only load checkpoints you trust.
assert FOLD_LEARNER_PATHS, (
    "No fold checkpoints found. Train the folds with notebook 01, or set "
    "FOLD_LEARNER_PATHS = {1: '.../best_learner.pkl', ...} in the config cell.")

predictors = []
for fold in sorted(FOLD_LEARNER_PATHS):
    predictors.append(_load_learner(FOLD_LEARNER_PATHS[fold]))
    print(f"Loaded fold {fold}: {FOLD_LEARNER_PATHS[fold]}")
print(f"Ensemble ready: {len(predictors)} fold model(s) for '{MODEL_KEY}'.")

## 5. Run inference

Pass a **list of learners** to `patch_inference` to soft-vote them: it averages each patch's class probabilities across the folds before the argmax, in one sliding-window pass. For each scan it reorders, resamples, and normalizes the input as in training, slides the Hann-blended patch grid, resamples the prediction back to the original grid, and post-processes (argmax + keep-largest-component). Predictions are written to `OUTPUT_DIR` as `<name>_pred.nii.gz` and returned in `predictions`.

- **`tta=USE_TTA`**: 8 axis-flip combinations averaged per patch. Tighter borders, ~8x compute, impractical on CPU.
- **`amp=USE_AMP`**: float16 forward pass; CUDA-only.

Pass `return_probabilities=True` for the averaged probability map instead of a mask.

In [ ]:
# A list of learners makes patch_inference soft-vote the folds; normalization comes from patch_config.
learners = predictors

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
predictions = patch_inference(
    learner=learners,
    config=patch_config,
    file_paths=NEW_CASES,
    save_dir=OUTPUT_DIR,
    progress=True,
    tta=USE_TTA,
    amp=USE_AMP,
)


def _pred_filename(input_path):
    """Output name patch_inference writes for a prediction: '<stem>_pred.nii[.gz]'."""
    p = Path(input_path)
    if p.suffix == ".gz" and p.stem.endswith(".nii"):
        return f"{p.stem[:-4]}_pred.nii.gz"
    if p.suffix == ".nii":
        return f"{p.stem}_pred.nii"
    return f"{p.stem}_pred.nii.gz"


print(f"\nWrote {len(predictions)} prediction(s) to {OUTPUT_DIR}/ "
      f"(soft-vote ensemble of {len(predictors)} folds).")
for p in NEW_CASES:
    print(f"  {p}  ->  {OUTPUT_DIR}/{_pred_filename(p)}")

## 6. Visualize a prediction

In [ ]:
from fastMONAI.vision_plot import *
import matplotlib.pyplot as plt

idx = 0
img_fn = NEW_CASES[idx]
# Same name patch_inference saved under, for both .nii and .nii.gz inputs.
pred_fn = Path(OUTPUT_DIR) / _pred_filename(img_fn)

# Default loader: no reorder/resample, so input and mask share the original grid.
img = MedImage.create(img_fn)
pred_mask = MedMask.create(pred_fn)

# Display aspect ratio only; does not alter data.
disp_spacing = patch_config.target_spacing

plane = 2  # 0 = sagittal, 1 = coronal, 2 = axial
sl = int(find_max_slice(pred_mask.data[0].cpu().numpy(), plane))
print(f"Predicted foreground voxels: {int(pred_mask.data.sum())}")
print(f"Showing plane={plane} (axial), slice={sl}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
img.show(ctx=axes[0], anatomical_plane=plane, slice_index=sl, voxel_size=disp_spacing)
axes[0].set_title("Input T1")
pred_mask.show(ctx=axes[1], anatomical_plane=plane, slice_index=sl, voxel_size=disp_spacing)
axes[1].set_title("Predicted mask")
show_mask_overlay(img, pred_mask, ctx=axes[2], anatomical_plane=plane,
                  slice_index=sl, voxel_size=disp_spacing, title="Overlay")
plt.tight_layout()
plt.show()

## 7. Optional: validate against ground truth

If a case has an expert segmentation, we score the prediction with the same metric functions notebook 01 uses, so the numbers are comparable. Runs only when `GT_PATH` points to a mask file.

In [ ]:
# Set to a ground-truth mask to score the first case, e.g.
# GT_PATH = "../nii_data/queen_square_data/vs_gk_1/vs_gk_1_seg_refT1.nii.gz"
GT_PATH = None

if GT_PATH:
    # Score the first prediction with the library's per-case panel.
    row = evaluate_segmentations([predictions[0]], [GT_PATH]).iloc[0]
    print(f"DSC:          {row['dsc']:.4f}")
    print(f"Sensitivity:  {row['sensitivity']:.4f}")
    print(f"Precision:    {row['precision']:.4f}")
    print(f"LDR:          {row['ldr']:.4f}")
    print(f"Signed RVE:   {row['rve']:.4f}")
    print(f"ASSD (mm):    {row['assd_mm']}")
    print(f"HD95 (mm):    {row['hd95_mm']}")
    print(f"NSD tau=1mm:  {row['nsd_tau1.0_mm']}")
    print(f"Spacing (mm): {row['spacing_mm']}  | status: {row['surface_status']}")
else:
    print("GT_PATH is None; skipping ground-truth evaluation.")

## 8. Running SegMamba on CPU (single-checkpoint demo)

This is a standalone demonstration of the CPU backend swap, not the ensemble path above. Only SegMamba needs it. Its Mamba blocks default to the `mamba_ssm`/`causal-conv1d` CUDA kernels, so on a CPU-only box you switch to `mamba_backend="mambamixer"` (pure PyTorch, needs `transformers`). A checkpoint trained with the default `mamba_ssm` backend loads into it with `strict=True`, no retraining.

In [ ]:
def _force_cpu_mamba_backend():
    """Make transformers' MambaMixer take its pure-PyTorch path instead of probing CUDA kernels (which crash on CPU or on an ABI-mismatched mamba_ssm build)."""
    import sys
    import transformers  # noqa: F401
    for modname in ("transformers.utils.import_utils", "transformers.utils"):
        mod = sys.modules.get(modname)
        if mod is not None:
            for fn in ("is_causal_conv1d_available", "is_mamba_ssm_available"):
                if hasattr(mod, fn):
                    setattr(mod, fn, (lambda *a, **k: False))


RUN_SEGMAMBA_CPU = False
SEGMAMBA_WEIGHTS = "models/best_segmamba.pth"  # single checkpoint on purpose: demos the CPU swap, not the fold ensemble

if RUN_SEGMAMBA_CPU:
    _force_cpu_mamba_backend()  # skips the mismatched CUDA kernel probe
    from models_segmamba.segmambav2 import SegMamba

    # Pure-PyTorch backend, runs on CPU.
    seg_model = SegMamba(
        in_chans=1, out_chans=2, depths=[2, 2, 2, 2],
        feat_size=[48, 96, 192, 384], hidden_size=768,
        mamba_backend="mambamixer",
    )
    # A mamba_ssm-trained checkpoint loads strict into the mambamixer model unchanged.
    from torch.nn.modules.utils import consume_prefix_in_state_dict_if_present
    sd = torch.load(SEGMAMBA_WEIGHTS, map_location="cpu")
    sd = sd["model"] if isinstance(sd, dict) and "model" in sd else sd
    consume_prefix_in_state_dict_if_present(sd, "_orig_mod.")  # official torch helper
    seg_model.load_state_dict(sd, strict=True)
    seg_model.eval()  # stays on CPU

    cpu_preds = patch_inference(
        learner=seg_model,
        config=patch_config,
        file_paths=NEW_CASES,
        save_dir=OUTPUT_DIR,
        progress=True,
        tta=False,   # TTA is impractical on CPU
    )
    print(f"SegMamba CPU: {len(cpu_preds)} prediction(s) written to {OUTPUT_DIR}/")
else:
    print("Set RUN_SEGMAMBA_CPU = True (with a SegMamba checkpoint) to run this path.")